## Week 2 Day 2

Our first Agentic Framework project!!

Prepare yourself for something ridiculously easy.

We're going to build a simple Agent system for generating cold sales outreach emails:
1. Agent workflow
2. Use of tools to call functions
3. Agent collaboration via Tools and Handoffs

## Before we start - some setup:


Please visit Sendgrid at: https://sendgrid.com/

(Sendgrid is a Twilio company for sending emails.)

If SendGrid gives you problems, see the alternative implementation using "Resend Email" in community_contributions/2_lab2_with_resend_email

Please set up an account - it's free! (at least, for me, right now).

Once you've created an account, click on:

Settings (left sidebar) >> API Keys >> Create API Key (button on top right)

Copy the key to the clipboard, then add a new line to your .env file:

`SENDGRID_API_KEY=xxxx`

And also, within SendGrid, go to:

Settings (left sidebar) >> Sender Authentication >> "Verify a Single Sender"  
and verify that your own email address is a real email address, so that SendGrid can send emails for you.


In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio



In [2]:
load_dotenv(override=True)

True

In [3]:
# Let's just check emails are working for you

def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("hieu0606sunny@gmail.com")  # Change to your verified sender
    to_email = To("hieupm.22ds@vku.udn.vn")  # Change to your recipient
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email()

202


### Did you receive the test email

If you get a 202, then you're good to go!

#### Certificate error

If you get an error SSL: CERTIFICATE_VERIFY_FAILED then students Chris S and Oleksandr K have suggestions:  
First run this: `!uv pip install --upgrade certifi`  
Next, run this:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

#### Other errors or no email

If there are other problems, you'll need to check your API key and your verified sender email address in the SendGrid dashboard

Or use the alternative implementation using "Resend Email" in community_contributions/2_lab2_with_resend_email

(Or - you could always replace the email sending code below with a Pushover call, or something to simply write to a flat file)

## Step 1: Agent workflow

In [4]:
instructions1 = "Bạn là một nhân viên kinh doanh làm việc cho ComplAI, \
một công ty cung cấp công cụ SaaS hỗ trợ đảm bảo tuân thủ tiêu chuẩn SOC2 \
và chuẩn bị cho việc kiểm toán dựa trên nền tảng AI. Bạn viết các email tiếp cận khách hàng (cold email) một cách chuyên nghiệp và nghiêm túc. "

instructions2 = "Bạn là một nhân viên kinh doanh hài hước và đầy sức hút làm việc cho ComplAI, \
một công ty cung cấp công cụ SaaS hỗ trợ đảm bảo tuân thủ tiêu chuẩn SOC2 \
và chuẩn bị cho việc kiểm toán dựa trên nền tảng AI. Bạn viết các email tiếp cận khách hàng hóm hỉnh, lôi cuốn và có khả năng cao nhận được phản hồi."

instructions3 = "Bạn là một nhân viên kinh doanh hài hước và đầy sức hút làm việc cho ComplAI, \
một công ty cung cấp công cụ SaaS hỗ trợ đảm bảo tuân thủ tiêu chuẩn SOC2 \
và chuẩn bị cho việc kiểm toán dựa trên nền tảng AI. Bạn viết các email tiếp cận khách hàng cực kỳ súc tích và đi thẳng vào vấn đề."

In [5]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-5-nano"
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-5-nano"
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-5-nano"
)

In [6]:

result = Runner.run_streamed(sales_agent1, input="Viết email chào hàng cho khách hàng mới.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Dưới đây là gợi ý email chào hàng cho khách hàng mới (bằng tiếng Việt, phong cách chuyên nghiệp và nghiêm túc). Bạn có thể chỉnh sửa theo tên người nhận và ngữ cảnh của mình.

Chủ đề (mẫu):
- Đảm bảo SOC 2 và chuẩn bị kiểm toán cho nền tảng AI của bạn dễ dàng hơn với ComplAI
- Giải pháp tự động tuân thủ SOC2 cho nền tảng AI và quy trình kiểm toán
- ComplAI giúp bạn rút ngắn thời gian chuẩn bị cho kiểm toán SOC2

Email:
Chào [Tên người nhận],

Tôi là [Tên] từ ComplAI. Chúng tôi cung cấp nền tảng SaaS giúp các tổ chức phát triển nền tảng AI/SaaS đảm bảo tuân thủ SOC 2 và chuẩn bị cho kiểm toán một cách nhanh chóng và có hệ thống.

Với ComplAI, đội ngũ của bạn có thể:
- Ánh xạ tự động các kiểm soát SOC 2 với hoạt động của nền tảng AI, giảm công sức thủ công.
- Thu thập và lưu trữ chứng cứ cho kiểm toán một cách liên tục và có tổ chức.
- Tự động sinh báo cáo kiểm toán và gắn nhãn chứng cứ cho auditor.
- Giám sát tuân thủ theo thời gian thực và nhận cảnh báo khi có bất kỳ lệch chuẩn nào.
- 

In [7]:
message = "Viết email chào hàng cho khách hàng mới."

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Dưới đây là một email chào hàng mẫu chuyên nghiệp cho khách hàng mới. Bạn có thể tùy chỉnh tên người nhận và tên công ty trước khi gửi.

Subject: ComplAI – Giải pháp tự động chuẩn bị SOC 2 và kiểm toán

Chào Anh/Chị [Tên], 

Tôi là [Tên], đại diện Kinh doanh tại ComplAI. Chúng tôi cung cấp nền tảng SaaS dựa trên AI giúp các tổ chức đảm bảo tuân thủ SOC 2 và chuẩn bị cho kiểm toán một cách nhanh chóng và có thể lặp lại.

Lợi ích chính khi sử dụng ComplAI:
- Tự động thu thập và sắp xếp bằng chứng tuân thủ từ nguồn dữ liệu hiện có (logs, policy docs, hệ thống quản trị).
- Gap analysis tự động và đề xuất remediation, giảm thời gian chuẩn bị và rủi ro thiếu chứng cứ.
- Báo cáo kiểm toán sẵn sàng xuất khẩu (PDF/SharePoint) và theo dõi readiness liên tục.
- Tích hợp dễ với các hệ thống hiện có (IAM, CI/CD, data stores) mà không ảnh hưởng đến hoạt động.

Chúng tôi đã hỗ trợ nhiều khách hàng ở quy mô khác nhau duy trì tuân thủ SOC 2 và chuẩn bị kiểm toán nhanh hơn.

Bạn có quan tâm lên lịch một

In [8]:
sales_picker = Agent(
    name="sales_picker",
    instructions="Bạn chọn email bán hàng (cold email) tốt nhất từ các phương án đã cho. \
    Hãy tưởng tượng bạn là khách hàng và chọn email mà bạn có khả năng phản hồi cao nhất. \
    Đừng đưa ra lời giải thích; chỉ phản hồi bằng nội dung của email được chọn.",
    model="gpt-5-nano"
)

In [9]:
message = "Viết email chào hàng cho khách hàng mới."

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


Best sales email:
Subject: SOC2 dễ như chơi — Demo nhanh 15 phút với ComplAI

Xin chào [Tên],

Mình là [Tên], đến từ ComplAI. Chúng tôi xây dựng nền tảng SaaS giúp các đội sản phẩm và bảo mật đảm bảo tuân thủ SOC 2 và chuẩn bị cho kiểm toán một cách thông minh, tự động và không đau đầu. Nói ngắn gọn: AI làm việc khó nhất cho bạn, bạn có thể quay lại thưởng cà phê và xem kết quả.

Tại sao bạn nên chú ý đến ComplAI:
- Tự động hóa chứng cứ: logs, cấu hình, quản lý truy cập và thay đổi được gom lại thành một bộ chứng cứ có thể gửi ngay cho auditor.
- Giám sát liên tục: đánh giá rủi ro theo thời gian thực để bạn biết khi nào cần hành động.
- Chuẩn bị kiểm toán sẵn sàng: ánh xạ kiểm soát, gói chứng cứ đầy đủ, có thể xuất ra đúng định dạng auditor yêu cầu.
- Tích hợp dễ với các công cụ bạn đang dùng (AWS/Azure/GCP, Jira, Slack, GitHub và nhiều nền tảng khác).
- Tiết kiệm thời gian và công sức – bạn có thể tập trung vào sản phẩm thay vì đống giấy tờ.

Khách hàng hiện tại nói gì? Chúng tôi đã g

Now go and check out the trace:

https://platform.openai.com/traces

## Part 2: use of tools

Now we will add a tool to the mix.

Remember all that json boilerplate and the `handle_tool_calls()` function with the if logic..

In [10]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-5-nano",
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-5-nano",
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-5-nano",
)

In [11]:
sales_agent1

Agent(name='Professional Sales Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='Bạn là một nhân viên kinh doanh làm việc cho ComplAI, một công ty cung cấp công cụ SaaS hỗ trợ đảm bảo tuân thủ tiêu chuẩn SOC2 và chuẩn bị cho việc kiểm toán dựa trên nền tảng AI. Bạn viết các email tiếp cận khách hàng (cold email) một cách chuyên nghiệp và nghiêm túc. ', prompt=None, handoffs=[], model='gpt-5-nano', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

## Steps 2 and 3: Tools and Agent interactions

Remember all that boilerplate json?

Simply wrap your function with the decorator `@function_tool`

In [12]:
@function_tool
def send_email(body: str):
    """ Gửi email với nội dung đã cho đến tất cả các khách hàng tiềm năng. """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("hieu0606sunny@gmail.com")  # Change to your verified sender
    to_email = To("hieupm.22ds@vku.udn.vn")  # Change to your recipient
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

### This has automatically been converted into a tool, with the boilerplate json created

In [13]:
# Let's look at it
send_email

FunctionTool(name='send_email', description='Gửi email với nội dung đã cho đến tất cả các khách hàng tiềm năng.', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000002684AF56480>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

### And you can also convert an Agent into a tool

In [14]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000002684AFC2E80>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

### So now we can gather all the tools together:

A tool for each of our 3 email-writing agents

And a tool for our function to send emails

In [15]:
description = "Viết email chào hàng cho khách hàng mới."


tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agent1', description='Viết email chào hàng cho khách hàng mới.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000002684A9AAA20>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent2', description='Viết email chào hàng cho khách hàng mới.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000002684AFC1BC0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 Functio

## And now it's time for our Sales Manager - our planning agent

In [16]:
# Improved instructions thanks to student Guillermo F.

instructions = """
Bạn là Quản lý Kinh doanh tại ComplAI. Mục tiêu của bạn là tìm ra email bán hàng (cold email) tốt nhất \
bằng cách sử dụng các công cụ sales_agent.
 
Hãy thực hiện cẩn thận các bước sau: 
1. Tạo bản thảo: Sử dụng cả ba công cụ sales_agent để tạo ra ba bản thảo email khác nhau. Không tiếp tục cho đến khi cả ba bản thảo đã sẵn sàng.

2. Đánh giá và Lựa chọn: Xem xét các bản thảo và chọn ra một email duy nhất mà bạn đánh giá là hiệu quả nhất.

3. Gửi email: Sử dụng công cụ send_email để gửi email tốt nhất đó (và chỉ duy nhất email đó) cho người dùng.
 
Các quy tắc quan trọng: 
- Bạn phải sử dụng ba công cụ sales_agent để tạo ba bản thảo khác nhau — không viết chúng của bạn.
- Bạn phải gửi một email duy nhất sử dụng công cụ send_email — không bao giờ gửi hơn một.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-5-nano")

message = "Gửi một email bán hàng (cold email) với lời chào là 'Thưa Giám đốc điều hành' (Dear CEO)"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Wait - you didn't get an email??</h2>
            <span style="color:#ff7800;">With much thanks to student Chris S. for describing his issue and fixes. 
            If you don't receive an email after running the prior cell, here are some things to check: <br/>
            First, check your Spam folder! Several students have missed that the emails arrived in Spam!<br/>Second, print(result) and see if you are receiving errors about SSL. 
            If you're receiving SSL errors, then please check out theses <a href="https://chatgpt.com/share/680620ec-3b30-8012-8c26-ca86693d0e3d">networking tips</a> and see the note in the next cell. Also look at the trace in OpenAI, and investigate on the SendGrid website, to hunt for clues. Let me know if I can help!
            </span>
        </td>
    </tr>
</table>

### And one more suggestion to send emails from student Oleksandr on Windows 11:

If you are getting certificate SSL errors, then:  
Run this in a terminal: `uv pip install --upgrade certifi`

Then run this code:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

Thank you Oleksandr!

## Remember to check the trace

https://platform.openai.com/traces

And then check your email!!


### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

With handoffs, control passes across



In [18]:

subject_instructions = "Bạn có thể viết tiêu đề cho một email bán hàng (cold email). \
Bạn sẽ nhận được một thông điệp và cần viết tiêu đề cho email sao cho có khả năng nhận được phản hồi cao nhất."

html_instructions = "Bạn có thể chuyển đổi nội dung email dạng văn bản thuần túy sang định dạng HTML. \
Bạn sẽ nhận được nội dung email dạng văn bản (có thể chứa định dạng markdown) \
và bạn cần chuyển đổi nó sang định dạng HTML với bố cục và thiết kế đơn giản, rõ ràng và lôi cuốn."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-5-nano")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Viết tiêu đề cho một email bán hàng (cold email)")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-5-nano")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Chuyển đổi nội dung email dạng văn bản sang định dạng HTML")


In [19]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Gửi một email với tiêu đề và nội dung HTML đã cho đến tất cả các khách hàng tiềm năng. """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("hieu0606sunny@gmail.com")  # Change to your verified sender
    to_email = To("hieupm.22ds@vku.udn.vn")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [20]:
tools = [subject_tool, html_tool, send_html_email]

In [21]:
tools

[FunctionTool(name='subject_writer', description='Viết tiêu đề cho một email bán hàng (cold email)', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000268465937E0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='html_converter', description='Chuyển đổi nội dung email dạng văn bản sang định dạng HTML', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000268489D34C0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool

In [22]:
instructions ="Bạn là người định dạng và gửi email. Bạn nhận được nội dung của một email cần gửi. \
Trước tiên, bạn sử dụng công cụ subject_writer để viết tiêu đề cho email, \
sau đó sử dụng công cụ html_converter để chuyển đổi nội dung sang định dạng HTML. \
Cuối cùng, bạn sử dụng công cụ send_html_email để gửi email với tiêu đề và nội dung HTML đó."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-5-nano",
    handoff_description="Chuyển đổi email sang định dạng HTML và gửi đi")


### Now we have 3 tools and 1 handoff

In [23]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Viết email chào hàng cho khách hàng mới.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000002684A9AAA20>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent2', description='Viết email chào hàng cho khách hàng mới.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000002684AFC1BC0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionT

In [25]:
# Improved instructions thanks to student Guillermo F.

sales_manager_instructions = """
Bạn là Quản lý Kinh doanh tại ComplAI. Mục tiêu của bạn là tìm ra email bán hàng (cold email) tốt nhất \
bằng cách sử dụng các công cụ sales_agent.
 
Hãy thực hiện cẩn thận các bước sau: 
1. Tạo bản thảo: Sử dụng cả ba công cụ sales_agent để tạo ra ba bản thảo email khác nhau. Không tiếp tục cho đến khi cả ba bản thảo đã sẵn sàng.

2. Đánh giá và Lựa chọn: Xem xét các bản thảo và chọn ra một email duy nhất dựa trên đánh giá của bạn về tính hiệu quả của nó.

3.Bàn giao (Handoff) để gửi: CHỈ chuyển bản thảo email chiến thắng cho 'Email Manager' agent. Email Manager sẽ chịu trách nhiệm định dạng và gửi đi.
 
Các quy tắc quan trọng: 
- Bạn phải sử dụng ba công cụ sales_agent để tạo ba bản thảo khác nhau — không viết chúng của bạn.
- Bạn phải bàn giao (Handoff) chính xác MỘT email cho Email Manager — tuyệt đối không nhiều hơn một..
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-5-nano")

message = "Gửi một email bán hàng (cold email) với lời chào 'Thưa Giám đốc điều hành' (Dear CEO) từ Alice."

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

### Remember to check the trace

https://platform.openai.com/traces

And then check your email!!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Can you identify the Agentic design patterns that were used here?<br/>
            What is the 1 line that changed this from being an Agentic "workflow" to "agent" under Anthropic's definition?<br/>
            Try adding in more tools and Agents! You could have tools that handle the mail merge to send to a list.<br/><br/>
            HARD CHALLENGE: research how you can have SendGrid call a Callback webhook when a user replies to an email,
            Then have the SDR respond to keep the conversation going! This may require some "vibe coding" 😂
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">This is immediately applicable to Sales Automation; but more generally this could be applied to  end-to-end automation of any business process through conversations and tools. Think of ways you could apply an Agent solution
            like this in your day job.
            </span>
        </td>
    </tr>
</table>

## Extra note:

Google has released their Agent Development Kit (ADK). It's not yet got the traction of the other frameworks on this course, but it's getting some attention. It's interesting to note that it looks quite similar to OpenAI Agents SDK. To give you a preview, here's a peak at sample code from ADK:

```
root_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.0-flash",
    description="Agent to answer questions about the time and weather in a city.",
    instruction="You are a helpful agent who can answer user questions about the time and weather in a city.",
    tools=[get_weather, get_current_time]
)
```

Well, that looks familiar!

And a student has contributed a customer care agent in community_contributions that uses ADK.